# Verifiable Backups with PQL.Assert

This notebook demonstrates a flexible workflow for managing **verifiable backups** of Power BI semantic models:

1. **Backup** a semantic model with timestamped naming
2. **Restore** the backup to a new semantic model in the same workspace
3. **Verify** the restored model's integrity using PQL.Assert automated tests

## Operation Modes

The notebook supports three operation modes controlled by the `operation` parameter:
- **"Backup"** - Create a timestamped backup file only
- **"Restore"** - Restore from an existing backup file and run verification tests
- **"Backup and Restore"** - Perform both backup and restore, then verify

By running PQL.Assert tests on the restored model, you can programmatically validate that the backup was successful and that all data quality checks pass.

## Prerequisites

- **semantic-link-labs** package
- **PQL.Assert** library installed in your semantic model
- Appropriate workspace permissions (Build or higher)

## Test Environment

This notebook uses the **BKUP** environment for backup verification tests.
Tests tagged with BKUP are specifically designed to validate restored semantic models.


## Step 1: Install Dependencies & Configure

Install required packages and define parameters for the backup and restore operation.

### Operation Modes

Set the `operation` parameter to control what actions to perform:
- **"Backup"** - Only create a backup file
- **"Restore"** - Only restore from an existing backup (requires `backup_file_path`)
- **"Backup and Restore"** - Perform both operations and run verification tests

In [ ]:
    operation = "Backup and Restore"                  # Operation mode: "Backup", "Restore", or "Backup and Restore"
    workspace = "verified-backups-research"           # Workspace name or ID
    dataset = "SampleModel"                           # Semantic model name or ID to backup
    restored_dataset_name = "SampleModel_Restored"    # Name for the restored model
    backup_file_path = ""                             # Path to backup file (required if operation is "Restore" only)
    test_environment = "BKUP"                         # Test environment: Fixed to "BKUP" for backup verification
    use_password = False                              # Set to True for password-protected backups
    key_vault_uri = ""                                # Key Vault URI (required if use_password=True)
    secret_name = ""                                  # Secret name in Key Vault (required if use_password=True)

In [ ]:
# ===== CONFIGURATION =====
# Assigned to json parameter for use later
params = {
    "operation": operation,
    "workspace": workspace,           
    "dataset": dataset,                           
    "restored_dataset_name": restored_dataset_name,
    "backup_file_path": backup_file_path,  
    "test_environment": test_environment,                        
    "use_password": use_password,                             
    "key_vault_uri": key_vault_uri,                                
    "secret_name": secret_name,                                 
}

# Validate operation parameter
valid_operations = ["Backup", "Restore", "Backup and Restore"]
if params["operation"] not in valid_operations:
    raise ValueError(f"operation must be one of: {valid_operations}")

# Validate test environment
if params["test_environment"] != "BKUP":
    raise ValueError("test_environment must be 'BKUP' for backup verification")

# Validate backup file path if restore-only mode
if params["operation"] == "Restore" and not params["backup_file_path"]:
    raise ValueError("backup_file_path is required when operation is 'Restore'")

print("✓ Configuration loaded")
print(f"  Operation: {params['operation']}")
print(f"  Test Environment: {params['test_environment']}")

## Step 2: Backup Semantic Model

Create a timestamped backup of the semantic model using semantic-link-labs.

In [ ]:
import sempy_labs as labs
import sempy.fabric as fabric
import pandas as pd
from datetime import datetime

# ───────────────────────────────────────────────────────────────────────────
# Step 2: Backup Semantic Model
# ───────────────────────────────────────────────────────────────────────────

# Extract parameters
operation = params.get("operation")
workspace = params.get("workspace")
dataset = params.get("dataset")
use_password = params.get("use_password", False)
key_vault_uri = params.get("key_vault_uri", "")
secret_name = params.get("secret_name", "")

# Resolve workspace and dataset IDs
workspace_id = fabric.resolve_workspace_id(workspace)
dataset_id = fabric.resolve_dataset_id(dataset, workspace=workspace)

# Initialize backup_name variable
backup_name = None

# Only perform backup if operation includes it
if operation in ["Backup", "Backup and Restore"]:
    # Generate timestamped backup name: workspace-id_dataset-id_YYYY_MM_DD_HH_MM_SS
    timestamp = datetime.now().strftime("%Y_%m_%d_%H_%M_%S")
    backup_name = f"{workspace_id}_{dataset_id}_{timestamp}"
    
    print(f"📦 Backing up semantic model: {dataset}")
    print(f"   Backup file: {backup_name}.abf")
    
    # Execute backup
    if use_password:
        backup_password = notebookutils.credentials.getSecret(key_vault_uri, secret_name)
        labs.backup_semantic_model(
            dataset=dataset,
            file_path=f"{backup_name}.abf",
            password=backup_password,
            allow_overwrite=True,
            apply_compression=True,
            workspace=workspace,
        )
        print(f"✓ Backup completed with password protection")
    else:
        labs.backup_semantic_model(
            dataset=dataset,
            file_path=f"{backup_name}.abf",
            allow_overwrite=True,
            apply_compression=True,
            workspace=workspace,
        )
        print(f"✓ Backup completed")
else:
    print(f"⏭  Skipping backup (operation={operation})")

## Step 3: Restore Semantic Model

Restore the backup to a new semantic model in the same workspace for verification.

In [ ]:
# ───────────────────────────────────────────────────────────────────────────
# Step 3: Restore Semantic Model
# ───────────────────────────────────────────────────────────────────────────

restored_dataset_name = params.get("restored_dataset_name", f"{dataset}_Restored")

# Only perform restore if operation includes it
if operation in ["Restore", "Backup and Restore"]:
    # Determine backup file path
    if operation == "Restore":
        # Use provided backup file path for restore-only mode
        backup_file = params.get("backup_file_path")
    else:
        # Use the backup file created in previous step
        backup_file = f"{backup_name}.abf"
    
    print(f"📂 Restoring semantic model from: {backup_file}")
    print(f"   New model name: {restored_dataset_name}")
    
    # Execute restore
    if use_password:
        backup_password = notebookutils.credentials.getSecret(key_vault_uri, secret_name)
        labs.restore_semantic_model(
            dataset=restored_dataset_name,
            file_path=backup_file,
            password=backup_password,
            allow_overwrite=True,
            ignore_incompatibilities=True,
            workspace=workspace,
            force_restore=True,
        )
    else:
        labs.restore_semantic_model(
            dataset=restored_dataset_name,
            file_path=backup_file,
            allow_overwrite=True,
            ignore_incompatibilities=True,
            workspace=workspace,
            force_restore=True,
        )
    
    print(f"✓ Restore completed: {restored_dataset_name}")
else:
    print(f"⏭  Skipping restore (operation={operation})")

## Step 4: Setup Test Infrastructure

Define helper functions for interacting with PQL.Assert via the XMLA endpoint.

## Step 5: Run Verification Tests

Execute all PQL.Assert tests for the specified environment on the restored model.

In [ ]:
# ───────────────────────────────────────────────────────────────────────────
# Step 4: Define PQL.Assert Test Helper Functions
# ───────────────────────────────────────────────────────────────────────────

def _retrieve_tests(workspace_id: str, dataset_id: str, environment: str) -> pd.DataFrame:
    """Retrieve PQL.Assert tests for a specific environment via XMLA.
    
    Args:
        workspace_id: GUID of the Fabric workspace
        dataset_id: GUID of the semantic model
        environment: Test environment name (e.g., "PROD", "DEV", "QA")
    
    Returns:
        DataFrame with columns: [Name], [Description], [PQLAssert_ImpersonatedUserName]
    """
    env_escaped = environment.replace('"', '""')
    dax = f'EVALUATE PQL.Assert.RetrieveTestsByEnvironmentV2("{env_escaped}")'
    return fabric.evaluate_dax(workspace=workspace_id, dataset=dataset_id, dax_string=dax)

def _execute_test(workspace_id: str, dataset_id: str, test_name: str) -> pd.DataFrame:
    """Execute a PQL.Assert test function without impersonation.
    
    Args:
        workspace_id: GUID of the Fabric workspace
        dataset_id: GUID of the semantic model
        test_name: Fully-qualified test function name
    
    Returns:
        DataFrame with columns: [TestName], [Expected], [Actual], [Passed]
    """
    dax = f"EVALUATE {test_name}()"
    return fabric.evaluate_dax(workspace=workspace_id, dataset=dataset_id, dax_string=dax)

def _execute_test_as_user(
    workspace_id: str,
    dataset_id: str,
    test_name: str,
    username: str,
) -> pd.DataFrame:
    """Execute a PQL.Assert test function with user impersonation (for RLS testing).
    
    Args:
        workspace_id: GUID of the Fabric workspace
        dataset_id: GUID of the semantic model
        test_name: Fully-qualified test function name
        username: UPN of the user to impersonate (e.g., "user@contoso.com")
    
    Returns:
        DataFrame with columns: [TestName], [Expected], [Actual], [Passed]
    """
    dax = f"EVALUATE {test_name}()"
    return labs.evaluate_dax_impersonation(
        dataset=dataset_id,
        dax_query=dax,
        user_name=username,
        workspace=workspace_id,
    )

print("✓ PQL.Assert helper functions defined")

In [ ]:
# ───────────────────────────────────────────────────────────────────────────
# Step 5: Run Verification Tests
# ───────────────────────────────────────────────────────────────────────────

# Only run tests if a restore was performed
if operation in ["Restore", "Backup and Restore"]:
    ENVIRONMENT = params.get("test_environment", "BKUP")
    print(f"\n🧪 Running PQL.Assert tests on restored model...")
    print(f"   Workspace: {workspace}")
    print(f"   Model: {restored_dataset_name}")
    print(f"   Environment: {ENVIRONMENT}\n")
    
    # Resolve the restored dataset ID
    restored_dataset_id = fabric.resolve_dataset_id(restored_dataset_name, workspace=workspace)
    
    # Retrieve tests via PQL.Assert.RetrieveTestsByEnvironmentV2
    try:
        tests_df = _retrieve_tests(workspace_id, restored_dataset_id, ENVIRONMENT)
    except Exception as exc:
        print(f"⚠  Could not retrieve tests. Is PQL.Assert installed in the model?")
        print(f"   Error: {exc}")
        tests_df = None
    
    if tests_df is None or tests_df.empty:
        print(f"ℹ  No tests found for environment '{ENVIRONMENT}'.")
        all_results = []
    else:
        print(f"✓ Found {len(tests_df)} test function(s) for environment '{ENVIRONMENT}'\n")
        
        all_results: list[pd.DataFrame] = []
        
        # Execute each test
        for idx, test_row in tests_df.iterrows():
            test_name: str = str(test_row.get("[Name]", test_row.iloc[0]))
            raw_user = test_row.get("[PQLAssert_ImpersonatedUserName]", "")
            
            # Handle impersonation (for RLS tests)
            if pd.isna(raw_user) or raw_user is None or str(raw_user).strip() in ("", "<NA>"):
                impersonated_user = ""
            else:
                impersonated_user = str(raw_user).strip()
            
            # Execute the test
            try:
                if impersonated_user:
                    print(f"  ▶ Executing '{test_name}' as '{impersonated_user}' (impersonated)")
                    result_df = _execute_test_as_user(
                        workspace_id, restored_dataset_id, test_name, impersonated_user
                    )
                else:
                    print(f"  ▶ Executing '{test_name}'")
                    result_df = _execute_test(workspace_id, restored_dataset_id, test_name)
            except Exception as exc:
                print(f"  ✗ Error: {exc}")
                result_df = pd.DataFrame({
                    "[TestName]": [test_name],
                    "[Expected]": ["Success"],
                    "[Actual]": [f"Error: {str(exc)}"],
                    "[Passed]": [False]
                })
            
            # Annotate results with context
            result_df.insert(0, "ImpersonatedUser", impersonated_user if impersonated_user else "")
            result_df.insert(0, "TestFunctionName", test_name)
            result_df.insert(0, "SemanticModelName", restored_dataset_name)
            result_df.insert(0, "SemanticModelId", restored_dataset_id)
            result_df.insert(0, "WorkspaceName", workspace)
            result_df.insert(0, "WorkspaceId", workspace_id)
            
            # Ensure ImpersonatedUser column stays as string type (not NA)
            result_df["ImpersonatedUser"] = result_df["ImpersonatedUser"].fillna("")
            
            all_results.append(result_df)
else:
    print(f"\n⏭  Skipping verification tests (operation={operation})")
    all_results = []

## Step 6: Review Test Results

Review the verification results to confirm the backup was restored successfully and all data quality checks pass.

In [ ]:
# ───────────────────────────────────────────────────────────────────────────
# Step 6: Review Test Results
# ───────────────────────────────────────────────────────────────────────────

# Only review results if tests were run
if operation not in ["Restore", "Backup and Restore"]:
    print(f"\n⏭  No test results to review (operation={operation})")
elif not all_results:
    print(
        "\n⚠  No test results were collected.\n"
        "   Verify that:\n"
        "   1. PQL.Assert is installed in the semantic model.\n"
        f"   2. Tests are defined for environment '{params.get('test_environment', 'BKUP')}'.\n"
        "   3. The notebook has Build (or higher) access to the workspace."
    )
else:
    # Combine all test results
    results_df = pd.concat(all_results, ignore_index=True)
    
    # Calculate summary statistics
    total = len(results_df)
    passed_col = "[Passed]" if "[Passed]" in results_df.columns else "Passed"
    passed = int(results_df[passed_col].sum()) if passed_col in results_df.columns else 0
    failed = total - passed
    
    # Display summary
    print("\n" + "="*80)
    print(f"Results  |  Total: {total}  |  ✅ Passed: {passed}  |  ❌ Failed: {failed}")
    print("="*80 + "\n")
    
    # Display the full results table
    display(results_df)
    
    # Final verification message
    if failed == 0:
        print("\n🎉 All tests passed! The backup was successfully verified.")
    else:
        print(f"\n⚠  {failed} test(s) failed. Review the results above for details.")
        raise Exception(f"Backup verification failed: {failed} out of {total} test(s) did not pass.")